In [ ]:
import requests

url = "https://www.iranjib.ir/showarchive/0/0"
response = requests.get(url)
print(response.text)

In [15]:
from bs4 import BeautifulSoup

soup = BeautifulSoup(response.text, 'html.parser')


In [ ]:
links_soup_page = soup.find_all('a')
url = set()

for link in links_soup:
     url.add(link.get('href'))

url = list(url)
gold_link = [link for link in url if "طلا" and "سکه" in link]

links_soup


In [7]:
gold_link = [link for link in url if "طلا" and "سکه" in link]
gold = gold_link[0]
gold

'https://www.iranjib.ir/shownews/145501/قیمت-جدید-سکه-و-طلا-امروز-سه-شنبه-1405-04-09/'

In [22]:
import jdatetime
import re
from bs4 import BeautifulSoup
import requests
url = "https://www.iranjib.ir/showarchive/0/0"
response = requests.get(url)
soup = BeautifulSoup(response.text, 'html.parser')
pagination = soup.find_all('table',attrs={'class':'pagination'})
page_links = []
with open('gold.txt','w',encoding='utf-8') as f:
    f.write('date,year,month,day,coin,gold\n')

for page in pagination:
    for link in page.find_all("a"):
        page_links.append(link["href"])
for page in page_links:
    response = requests.get(page)
    soup = BeautifulSoup(response.text, 'html.parser')
    links_soup_page = soup.find_all('a')
    url = set()
    for link in links_soup_page:
        url.add(link.get('href'))
    url = list(url)
    gold_link = [link for link in url if "طلا" and "سکه" in link]

    with open('gold.txt','a',encoding='utf-8') as f:
         for link in gold_link:
            try:
                page = requests.get(link)
                page_soup = BeautifulSoup(page.text , "html.parser")
                date_soup = page_soup.find_all('td',attrs={"style":"width:60%"})[0].get_text()
                day, month_name, year = re.search(r'(\d+)\s+(\S+)\s+(\d+)', date_soup).groups()
                months = {
                    "فروردین": 1, "اردیبهشت": 2, "خرداد": 3, "تیر": 4,
                    "مرداد": 5, "شهریور": 6, "مهر": 7, "آبان": 8,
                    "آذر": 9, "دی": 10, "بهمن": 11, "اسفند": 12
                }
                date_Shamsi = jdatetime.date(int(year),int(months[month_name]),int(day)).strftime("%Y/%m/%d")


                table = page_soup.find(attrs={"class":"nij"})
                for row in table.find_all('tr') :
                    if "سکه گرمی" in row.text:
                        price_coin =int( row.find_all('td')[1].get_text(strip=True).replace(',',''))
                    if "طلای 18 عیار" in row.text:
                        price_gold = int( row.find_all('td')[1].get_text(strip=True).replace(',',''))

                f.write(f'{date_Shamsi} , {year} , {month_name} , {day} , {price_coin} , {price_gold}\n') 
         
            except Exception as e:
               print( f'error {link} message {e}')

error https://www.iranjib.ir/shownews/145153/ریزش-سریالی-قیمت-سکه-و-دلار/ message 'NoneType' object has no attribute 'find_all'


In [1]:
import sqlite3
import jdatetime
import re
from bs4 import BeautifulSoup
import requests
conn = sqlite3.Connection('gold_coin.db')
cursor = conn.cursor()
state_Create_table ='''
                              create table if not exists gold 
                                    (id, Date_Shamsi , year , MonthName , Days ,Price_coin , Price_Gold)

'''
cursor.execute(state_Create_table)
state_Insert_table = '''
                                   insert into gold  values (?,?,?,?,?,?,?)

                                   '''
url = "https://www.iranjib.ir/showarchive/0/0"
response = requests.get(url)
soup = BeautifulSoup(response.text, 'html.parser')
pagination = soup.find_all('table',attrs={'class':'pagination'})
page_links = []


for page in pagination:
    for link in page.find_all("a"):
        page_links.append(link["href"])
for page in page_links:
    response = requests.get(page)
    soup = BeautifulSoup(response.text, 'html.parser')
    links_soup_page = soup.find_all('a')
    url = set()
    for link in links_soup_page:
        url.add(link.get('href'))
    url = list(url)
    gold_link = [link for link in url if "طلا" and "سکه" in link]

  
    for i, link in enumerate(gold_link):
           try:
                page = requests.get(link)
                page_soup = BeautifulSoup(page.text , "html.parser")
                date_soup = page_soup.find_all('td',attrs={"style":"width:60%"})[0].get_text()
                day, month_name, year = re.search(r'(\d+)\s+(\S+)\s+(\d+)', date_soup).groups()
                months = {
                    "فروردین": 1, "اردیبهشت": 2, "خرداد": 3, "تیر": 4,
                    "مرداد": 5, "شهریور": 6, "مهر": 7, "آبان": 8,
                    "آذر": 9, "دی": 10, "بهمن": 11, "اسفند": 12
                }
                date_Shamsi = jdatetime.date(int(year),int(months[month_name]),int(day)).strftime("%Y/%m/%d")


                table = page_soup.find(attrs={"class":"nij"})
                for row in table.find_all('tr') :
                    if "سکه گرمی" in row.text:
                        price_coin =int( row.find_all('td')[1].get_text(strip=True).replace(',',''))
                    if "طلای 18 عیار" in row.text:
                        price_gold = int( row.find_all('td')[1].get_text(strip=True).replace(',',''))
                  
                cursor.execute(state_Insert_table,(i,
                                                 date_Shamsi,
                                                 year,
                                                  month_name,
                                                  day,
                                                  price_coin,
                                                  price_gold
                                                           
                                                           
                                                )
                                                           )
                conn.commit
           except Exception as e:
                  print( f'error: {link} message: {e}')


error: https://www.iranjib.ir/shownews/145153/ریزش-سریالی-قیمت-سکه-و-دلار/ message: 'NoneType' object has no attribute 'find_all'


In [9]:
import jdatetime
import re
from bs4 import BeautifulSoup
import requests
url = "https://www.iranjib.ir/showarchive/0/0"
response = requests.get(url)
soup = BeautifulSoup(response.text, 'html.parser')
pagination = soup.find_all('table',attrs={'class':'pagination'})
page_links = []
for page in pagination:
     for link in page.find_all('a'):
          page_links.append(link.get('href'))


In [13]:
link


<a class="jaxlink" href="https://www.iranjib.ir/jax/showarchive.php?p=1&amp;_id=12" wrapper="showarchivetarget"><button class="paginbutton">12</button></a>